In [0]:
create or replace temporary table video_history as 
select ChannelName, Title, VideoID, PublishedAt, v.ViewCount, v.LikeCount, v.CommentCount, v.ScrapedAt
from youtube_lakehouse.silver.fact_videos_history v
inner join (select distinct channelID, channelName from youtube_lakehouse.silver.dim_channels_history ) c
on v.ChannelID = c.ChannelID
--where ChannelName = 'WildLens by Abrar'
union all
select channel_title, video_title, video_id, published_at, cumulative_views, cumulative_likes, cumulative_comments, snapshot_date
from youtube_lakehouse.silver.fact_video_daily_snapshots
--where channel_title = 'WildLens by Abrar'
order by channelname, PublishedAt,  v.ScrapedAt;

In [0]:
create or replace temporary table daily_deltas as 
with daily_snapshot as (
    select
    ChannelName,
    Title,
    videoid,
    date(publishedAt),
    date(ScrapedAt) as snapshot_date,
    max(ViewCount) as maxViews,
    max(LikeCount) as maxLikes,
    max(CommentCount) as maxComment    
from video_history
group by Title, ChannelName, videoid,  date(publishedAt),date(ScrapedAt)
),
daily_deltas as (
  select 
    Title,
    publishedAt,
    snapshot_date,
    maxViews,
    maxViews - lag(maxViews, 1) over (
      partition by Title 
      order by snapshot_date
    ) as DoDViews_Delta,
    maxLikes,
    maxLikes - lag(maxLikes, 1) over (
      partition by Title 
      order by snapshot_date
    ) as DoDLikes_Delta,
    maxComment,
    maxComment - lag(maxComment, 1) over (
      partition by Title 
      order by snapshot_date
    ) as DoDComment_Delta
  from daily_snapshot
)
select 
  Title,
  publishedAt,
  snapshot_date,
  maxViews,
  DoDViews_Delta,
  DoDLikes_Delta,
  DoDComment_Delta
from daily_deltas
--where   publishedat > date('2026-08-20')
--where snapshot_date > date('2026-06-20')
order by publishedAt desc, Title, snapshot_date;

In [0]:
create table if not exists youtube_lakehouse.gold.video_history as 
select Title,
  publishedAt,
  snapshot_date,
  maxViews,
  DoDViews_Delta,
  DoDLikes_Delta,
  DoDComment_Delta
from daily_deltas;